In [1]:
import os
import json
import pandas as pd
from openai import OpenAI
from getpass import getpass
from pathlib import Path
from datetime import datetime

In [2]:
if not os.getenv("OPENAI_API_KEY_QUESTIONS_GENERATOR"):
    os.environ["OPENAI_API_KEY_QUESTIONS_GENERATOR"] = getpass("Enter your OpenAI key to generate questions: ")

In [3]:
# get all metadata from last json file

STATS_DIR = Path(os.path.join(os.path.dirname(os.getcwd()), "rag", "stats"))

most_recent_file = max(
    (f for f in STATS_DIR.iterdir() if f.suffix == ".json"),
    key=lambda f: f.stat().st_mtime
)

with open(most_recent_file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [4]:
def get_all_languages(metadata_dict):
    return list({ref.get("lex_lang") for ref in metadata_dict.get("refs")})

def generate_question_from_ai(doc_url, langs):
    prompts = {
        "fr": "Envoie moi simplement une question courte qui peut être répondue en lisant ce document.",
        "en": "Send me a short question that can be answered by reading this document."
    }

    if len(langs) == 1:
        prompt = prompts.get(langs[0])
    else:
        prompt = prompts.get("fr")
        prompt += "\nEcris la question dans la langue du document transmis."

    print(prompt) # TODO : supprimer quand ok

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_QUESTIONS_GENERATOR"))
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_file", "file_url": doc_url}
                ]
            }
        ]
    )
    return response.output_text

In [5]:
# generate pairs of question -> doc

test_dataset = []

nb_files = 0 # TODO : supprimer limite quand ok

for content, metadata_dict in data.items():
    # TODO : supprimer limite quand ok
    if nb_files == 3:
        break
    if metadata_dict.get("content_format") in ["pdf", "docx"]: # FIXME : need to exclude only txt for the moment
        if content != "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/LEX-5.1.0.3.pdf": # FIXME : exclude big file (448 pages)
            langs = get_all_languages(metadata_dict)
            question = generate_question_from_ai(content, langs)
            pair = {
                "src": "openai",
                "langs": langs,
                "question": question, # TODO : transformer en liste si necessaire
                "doc_url": content,
                "doc_id": metadata_dict.get("doc_id")
            }
            test_dataset.append(pair)
            print(f"Question is '{question}' for doc_url '{content}'.") # TODO : a supprimer quand ok
            nb_files += 1 # TODO : supprimer limite quand ok

df_test_dataset = pd.DataFrame(test_dataset)

Envoie moi simplement une question courte qui peut être répondue en lisant ce document.
Question is Quelle est la mission principale des écoles polytechniques fédérales (EPF) selon la loi sur les EPF? for doc_url 'https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/cc/1993/210_210_210/20250501/fr/pdf-a/fedlex-data-admin-ch-eli-cc-1993-210_210_210-20250501-fr-pdf-a.pdf'
Send me a short question that can be answered by reading this document.
Question is What are the main responsibilities of the institutions within the ETH Domain according to the ETH Act? for doc_url 'https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/cc/1993/210_210_210/20250501/en/pdf-a/fedlex-data-admin-ch-eli-cc-1993-210_210_210-20250501-en-pdf-a.pdf'
Envoie moi simplement une question courte qui peut être répondue en lisant ce document.
Question is Quelles sont les écoles polytechniques fédérales mentionnées dans le document ? for doc_url 'https://fedlex.data.admin.ch/filestore/fedlex.data.ad

In [6]:
# load openai test dataset in json file

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
openai_test_dataset_filename = f"{timestamp}_openai_test_dataset.json"
df_test_dataset.to_json(openai_test_dataset_filename, orient="records", indent=4, force_ascii=False)

In [7]:
# TODO : merger les questions venant des differentes sources (openai, webmaster, service_desk, service_etudiants, ...)